<details>
<summary><b>Info</b></summary>

**Last Execution:** 2026-07-25

| Package | Version |
|---------|---------|
| **nnsight** | **0.8** |
| Python | 3.12.13 |
| torch | 2.13.0+cu126 |
| transformers | 5.15.0 |

</details>


# Getting Activations

This tutorial assumes you've already been through the main [Walkthrough](../tutorials/tutorials/get_started/walkthrough.ipynb).

Reading intermediate values from a model's forward pass is one of the most fundamental operations in nnsight. This page covers how to access a module's output, its inputs, and how to persist those values so you can use them after the trace ends.

## Setup

We load GPT-2 with `TransformersModel`.

<details><summary><b>More on models</b></summary>

For details on <code>TransformersModel</code>, <code>device_map</code>, dispatching, and loading options, see the <a href="../documentation/modeling/transformers.md">transformers model guide</a> and the <a href="5_loading.ipynb">Loading a Model</a> tutorial.

</details>

In [1]:
from nnsight.modeling.transformers import TransformersModel

model = TransformersModel("openai-community/gpt2", device_map="auto", dispatch=True)

/home/localjadenfk/miniconda3/envs/ndif2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Getting a Layer Output

Open a tracing context with `model.trace(...)`, then read a module's forward-pass return value with `.output`.

Inside a trace, a value like `.output` is only a proxy — a placeholder for something the forward pass hasn't produced yet — so it isn't available once the `with` block exits. `.save()` marks a value to keep, and bound to a variable it survives the trace so you can read it afterward. This is your first look at `.save()`; it shows up throughout these tutorials.

In [2]:
with model.trace("The Eiffel Tower is in the city of"):

    hidden_states = model.transformer.h[-1].output.save()

print(hidden_states.shape)

torch.Size([1, 10, 768])


<details class="admonition warning">
<summary>Always <code>.save()</code> and bind to a variable</summary>

A traced `with` block does not run in place — its body runs in a worker that only returns values you explicitly mark. `.save()` marks a value to survive the block, and it comes back *by the variable name you bind it to*, so write `hidden_states = ....save()`. A bare `model.transformer.h[-1].output.save()` on its own line is marked but has no name to return under, so it is silently lost.

Calling `.save()` outside a trace now raises `ValueError` (in earlier versions it was a silent no-op).

</details>

## Getting a Layer Input

Use `.input` to read the first positional argument passed into a module.

In [3]:
with model.trace("The Eiffel Tower is in the city of"):

    layer_input = model.transformer.h[0].input.save()

print(layer_input.shape)

torch.Size([1, 10, 768])


## `.input` vs `.inputs`

`.input` is a convenience returning just the first positional argument. `.inputs` returns the full call signature as an `(args, kwargs)` tuple — every positional argument and every keyword argument the module was called with.

In [4]:
with model.trace("The Eiffel Tower is in the city of"):

    detailed_inputs = model.transformer.h[0].inputs.save()

args, kwargs = detailed_inputs
print(f"Positional args: {len(args)}")
print(f"Keyword args: {list(kwargs.keys())}")

Positional args: 4
Keyword args: ['encoder_attention_mask', 'use_cache', 'position_ids']


<details class="admonition warning">
<summary>Access modules in forward-pass order</summary>

Within a single invoke, request modules in the order they run. Reading a later module and then an earlier one deadlocks and raises `OutOfOrderError` once the forward pass finishes past the earlier module. To read modules out of order, use separate invokes (see the batching guide).

</details>

## `nnsight.save()` vs `.save()`

There are two equivalent ways to save a value. The **preferred** form is the function `nnsight.save(...)`, which works on any object:

In [5]:
import nnsight

with model.trace("The Eiffel Tower is in the city of"):

    hidden_states = nnsight.save(model.transformer.h[-1].output)

print(hidden_states.shape)

torch.Size([1, 10, 768])


<details class="admonition note">
<summary><code>nnsight.save()</code> vs <code>obj.save()</code></summary>

`obj.save()` relies on a C extension that mounts a `.save()` method onto every Python object at import time. It works in most cases, but `nnsight.save()` is safer: it is unaffected if a class defines its own `.save()` method (which would shadow nnsight's version). **Prefer `nnsight.save()` for plain Python values** like ints, lists, and dicts.

</details>

## Getting Multiple Layer Outputs

Collect hidden states from every layer in a single forward pass. Save the **container** and put the raw per-layer outputs into it — do not `.save()` the individual elements.

In [6]:
with model.trace("The Eiffel Tower is in the city of"):

    hidden_states_per_layer = nnsight.save(
        [layer.output for layer in model.transformer.h]
    )

for i, hs in enumerate(hidden_states_per_layer):
    print(f"Layer {i}: {hs.shape}")

Layer 0: torch.Size([1, 10, 768])
Layer 1: torch.Size([1, 10, 768])
Layer 2: torch.Size([1, 10, 768])
Layer 3: torch.Size([1, 10, 768])
Layer 4: torch.Size([1, 10, 768])
Layer 5: torch.Size([1, 10, 768])
Layer 6: torch.Size([1, 10, 768])
Layer 7: torch.Size([1, 10, 768])
Layer 8: torch.Size([1, 10, 768])
Layer 9: torch.Size([1, 10, 768])
Layer 10: torch.Size([1, 10, 768])
Layer 11: torch.Size([1, 10, 768])


<details class="admonition tip">
<summary>Saving collections</summary>

Save the list itself (here via `nnsight.save([...])`); a saved container comes back with its contents. If you instead `.save()` each element the marks have no name to return under, and if you leave the list unsaved it never comes back at all.

</details>

## Using Saved Outputs

Saved activations are real tensors, so you can run any PyTorch operation on them after the tracing context exits.

In [7]:
with model.trace("The Eiffel Tower is in the city of"):

    logits = model.lm_head.output.save()

predicted_token = logits[0, -1].argmax(dim=-1)
print(f"Predicted next token: {model.tokenizer.decode(predicted_token)}")

Predicted next token:  Paris
